# Lightweight Reliability-Adaptive Federated Multimodal Intelligence for Financial Fraud Detection and Uncertainty-Aware Risk Assessment

**Dataset:** [colabsss/financial-fraud-multimodal-dataset](https://www.kaggle.com/datasets/colabsss/financial-fraud-multimodal-dataset)

| Stage | Content |
|---|---|
| 1 | Company-ID integration, median imputation + robust scaling, structure-preserving text cleaning, category-aware term normalisation |
| 2 | **Contribution 1** – FT-Transformer-lite, frozen FinBERT, TF-IDF + category embeddings, reliability-gated fusion |
| 3 | Federated client initialisation (company-level partitioning) |
| 4 | Local training (AdamW, class-weighted BCE, component-wise updates) |
| 5 | **Contribution 2** – reliability-adaptive, modality-aware, clipped aggregation |
| 6 | Prediction, temperature scaling, cost-based thresholding |
| 7 | Lightweight explainability (permutation importance, linear lexical baseline, modality ablation) |
| 8 | **Contribution 3** – uncertainty-aware risk ranking, screening, reports, optional portfolio integration |

The column-detection cell auto-discovers the files/columns. If your copy of the dataset uses different names, override them in the `CFG` dictionary (`text_cols`, `lex_cols`, `num_cols`, `label_col`).

## 0. Setup and configuration

In [ ]:
import os, re, glob, html, json, math, copy, random, hashlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, precision_score, recall_score, average_precision_score,
                             roc_auc_score, matthews_corrcoef, brier_score_loss,
                             confusion_matrix, precision_recall_curve)
warnings.filterwarnings('ignore')

CFG = dict(
    seed=42,
    data_dir=None,                 # None -> /kaggle/input or kagglehub download
    out_dir='/kaggle/working/fraud_outputs' if os.path.isdir('/kaggle/working') else './fraud_outputs',
    label_col=None, text_cols=[], lex_cols=[], num_cols=[],      # optional manual overrides
    # splits (company level)
    test_size=0.15, calib_size=0.10, val_size=0.10,
    # Stage 2
    use_finbert=True, finbert_name='ProsusAI/finbert', max_chunk_tokens=256, max_chunks=4,
    ft_dim=16, ft_layers=2, ft_heads=2, d_model=64, svd_dim=32, tfidf_max_features=500,
    # Stage 3-5
    n_clients=5, noniid_alpha=1.0, rounds=25, patience=6, local_epochs=2, batch=64,
    lr=2e-3, wd=1e-4, mod_dropout=0.15, w_clip=0.40,
    simulate_modality_gaps=True,   # clients differ in text/lexical availability (reliability demo)
    text_gap_rates=[0.0, 0.0, 0.10, 0.25, 0.45],
    # Stage 6/8
    cost_fn=5.0, cost_fp=1.0, band=(0.40, 0.60), high_cut=0.75, med_cut=0.50,
    price_csv=None,                # optional: CSV with Company_ID, Date, Return for Stage 8.4
)
os.makedirs(CFG['out_dir'], exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_all(CFG['seed'])
print('Device:', DEVICE)

## Data collection and Stage 1.1 – Company-ID-based data integration

In [ ]:
def find_dataset():
    if CFG['data_dir']: return CFG['data_dir']
    for p in ['/kaggle/input/financial-fraud-multimodal-dataset', '/kaggle/input']:
        if os.path.isdir(p) and (glob.glob(p + '/**/*.csv', recursive=True) or glob.glob(p + '/**/*.xlsx', recursive=True)):
            return p
    import kagglehub
    return kagglehub.dataset_download('colabsss/financial-fraud-multimodal-dataset')

def read_any(p):
    if p.lower().endswith('.csv'):
        try: return pd.read_csv(p)
        except UnicodeDecodeError: return pd.read_csv(p, encoding='latin-1')
    return pd.read_excel(p)

DATA_DIR = find_dataset()
files = sorted(glob.glob(DATA_DIR + '/**/*.csv', recursive=True) + glob.glob(DATA_DIR + '/**/*.xlsx', recursive=True))
frames = {os.path.relpath(p, DATA_DIR): read_any(p) for p in files}
for k, v in frames.items():
    print(f'{k:50s} shape={v.shape}  cols={list(v.columns)[:10]}')

def squash(c): return re.sub(r'[^a-z0-9]', '', str(c).lower())
def find_id(df):
    for c in df.columns:
        if squash(c) in ('companyid', 'firmid', 'companycode', 'cid'): return c
    return None

dfs = []
for k, d in frames.items():
    idc = find_id(d)
    if idc is None: print('Skipping (no Company_ID):', k); continue
    d = d.rename(columns={idc: 'Company_ID'}).copy()
    d['Company_ID'] = d['Company_ID'].astype(str).str.strip()
    dfs.append(d.drop_duplicates())

def merge_all(dfs):
    out = dfs[0]
    for d in dfs[1:]:
        keys = ['Company_ID'] + [c for c in out.columns if c != 'Company_ID' and re.search(r'year|period|date', str(c).lower()) and c in d.columns]
        dup = [c for c in d.columns if c in out.columns and c not in keys]
        out = out.merge(d.drop(columns=dup), on=keys, how='outer')
    return out

df = merge_all(dfs).reset_index(drop=True)

# target
label_col = CFG['label_col']
if label_col is None:
    cands = [c for c in df.columns if re.search(r'fraud', str(c).lower()) and re.search(r'label|target|flag|class|is', str(c).lower())] \
            or [c for c in df.columns if 'fraud' in str(c).lower()] or [c for c in df.columns if re.search(r'label|target', str(c).lower())]
    label_col = cands[0]
y_raw = df[label_col]
if y_raw.dtype == object:
    y_raw = y_raw.astype(str).str.strip().str.lower().map(lambda s: 1 if s in ('1', 'fraud', 'yes', 'true', 'y') else (0 if s in ('0', 'no fraud', 'no', 'false', 'n', 'non-fraud', 'nonfraud') else np.nan))
df['Fraud_Label'] = pd.to_numeric(y_raw, errors='coerce')
df = df.dropna(subset=['Fraud_Label']).reset_index(drop=True)
df['Fraud_Label'] = df['Fraud_Label'].astype(int)
if label_col != 'Fraud_Label': df = df.drop(columns=[label_col])
print('Merged:', df.shape, '| companies:', df.Company_ID.nunique(), '| fraud rate: %.3f' % df.Fraud_Label.mean())

### Column role detection (numerical / narrative / lexical)

In [ ]:
SKIP_NUM = re.compile(r'^(year|fiscalyear|period|date|index|unnamed.*|companyid|fraudlabel)$')
obj = [c for c in df.columns if df[c].dtype == object and c not in ('Company_ID',)]
for c in obj:                                           # numeric-looking strings -> numbers
    conv = pd.to_numeric(df[c], errors='coerce')
    if df[c].notna().any() and conv.notna().sum() >= 0.9 * df[c].notna().sum(): df[c] = conv
def avg_len(s): return s.dropna().astype(str).str.len().mean() if s.notna().any() else 0
obj = [c for c in df.columns if df[c].dtype == object and c != 'Company_ID']

TEXT_COLS = CFG['text_cols'] or [c for c in obj if avg_len(df[c]) >= 150]
LEX_COLS = CFG['lex_cols'] or [c for c in obj if c not in TEXT_COLS and avg_len(df[c]) < 150
                              and re.search(r'term|word|lexic|keyword|sentiment|strateg|governance|financial|risk', str(c).lower())]
NUM_COLS = CFG['num_cols'] or [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and not SKIP_NUM.match(squash(c))]
print('Numerical :', len(NUM_COLS), NUM_COLS[:12], '...')
print('Narrative :', TEXT_COLS)
print('Lexical   :', LEX_COLS)
assert len(NUM_COLS) > 0, 'No numerical columns detected - set CFG["num_cols"]'
assert len(TEXT_COLS) > 0, 'No narrative columns detected - set CFG["text_cols"]'

CATS = ['financial', 'governance', 'sentiment', 'risk', 'strategy']
def lex_cat(name):
    n = str(name).lower()
    for cat, pat in [('sentiment', 'sentiment|tone|polarity'), ('risk', 'risk'), ('governance', 'govern|audit|board'),
                     ('strategy', 'strateg'), ('financial', 'financ|profit|revenue|debt')]:
        if re.search(pat, n): return cat
    return 'other'

## Stage 1.3 / 1.4 – Structure-preserving text cleaning and category-aware term normalisation
Negations, monetary amounts, percentages and risk vocabulary are preserved (no stop-word removal, no stemming).

In [ ]:
UNIT = {'million': 'm', 'mn': 'm', 'm': 'm', 'billion': 'b', 'bn': 'b', 'b': 'b', 'thousand': 'k', 'k': 'k'}
def clean_text(s):
    if not isinstance(s, str): return ''
    s = html.unescape(s)
    s = re.sub(r'https?://\S+|www\.\S+|<[^>]+>', ' ', s)
    s = s.replace('\u2019', "'").replace('\u2018', "'").replace('\u201c', '"').replace('\u201d', '"')
    s = re.sub(r'(?i)\$\s*(\d[\d,]*\.?\d*)\s*(million|billion|thousand|mn|bn|m|b|k)\b',
               lambda m: '$' + m.group(1).replace(',', '') + UNIT[m.group(2).lower()], s)
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s\.\,\;\:\%\$\-\'/\(\)]", ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

SYN = {'net_income': 'profit', 'net_profit': 'profit', 'earnings': 'profit', 'lawsuit': 'litigation', 'lawsuits': 'litigation',
       'litigations': 'litigation', 'corp_governance': 'governance', 'debts': 'debt', 'leverage_ratio': 'leverage'}
def norm_terms(cell, cat):
    if cell is None or (isinstance(cell, float) and np.isnan(cell)): return []
    parts = re.split(r'[;,|\n]+', str(cell))
    out = []
    for p in parts:
        t = re.sub(r'[^a-z0-9]+', '_', p.strip().lower()).strip('_')
        if t and t not in ('nan', 'none'): out.append(f'{cat}:{SYN.get(t, t)}')
    return out

LEXICON = {   # fallback: derive the five lexical fields from the narrative text if the dataset has no term columns
    'financial': ['revenue', 'profit', 'loss', 'debt', 'leverage', 'liquidity', 'cash flow', 'margin', 'impairment', 'write-off', 'restatement', 'receivables', 'inventory'],
    'governance': ['board', 'independent director', 'audit', 'auditor', 'internal control', 'compliance', 'related party', 'disclosure', 'whistleblower', 'oversight'],
    'sentiment': ['strong', 'growth', 'confident', 'improve', 'decline', 'weak', 'concern', 'uncertain', 'adverse', 'pressure', 'optimistic'],
    'risk': ['litigation', 'regulatory', 'default', 'penalty', 'investigation', 'fraud', 'sanction', 'volatility', 'going concern', 'material weakness'],
    'strategy': ['expansion', 'acquisition', 'diversification', 'innovation', 'restructuring', 'market share', 'investment', 'partnership', 'digital']}
LEX_RE = {c: re.compile(r'\b(' + '|'.join(re.escape(t) for t in sorted(v, key=len, reverse=True)) + r')\b') for c, v in LEXICON.items()}

# ---- apply cleaning
for c in TEXT_COLS: df[c + '__clean'] = df[c].map(clean_text)
all_text = df[[c + '__clean' for c in TEXT_COLS]].agg(' '.join, axis=1)
if LEX_COLS:
    LEX_FIELDS = []
    for c in LEX_COLS:
        cat = lex_cat(c); name = f'lex__{c}'
        df[name] = df[c].map(lambda v, cat=cat: ' '.join(norm_terms(v, cat))); LEX_FIELDS.append(name)
else:
    print('No lexical columns found -> deriving category lexical fields from narrative text.')
    LEX_FIELDS = []
    for cat in CATS:
        name = f'lex__{cat}'
        df[name] = all_text.map(lambda t, cat=cat: ' '.join(f'{cat}:' + m.replace(' ', '_').replace('-', '_') for m in LEX_RE[cat].findall(t)))
        LEX_FIELDS.append(name)
print('Lexical fields:', LEX_FIELDS)
display(df[[TEXT_COLS[0], TEXT_COLS[0] + '__clean'] + LEX_FIELDS[:2]].head(3))

## Stage 1.2 – Company-level split, median imputation, robust scaling, missingness indicators
Splits are made **by company** (no leakage); all statistics are fitted on the training companies only.
`train` → local federated training, `val` → reliability / model selection / threshold, `calib` → temperature scaling, `test` → final report.

In [ ]:
comp = df.groupby('Company_ID')['Fraud_Label'].max()
ids, yy = comp.index.values, comp.values
def strat_split(a, ya, size, seed):
    try: return train_test_split(a, ya, test_size=size, stratify=ya, random_state=seed)
    except ValueError: return train_test_split(a, ya, test_size=size, random_state=seed)
rest, test_ids, y_rest, _ = strat_split(ids, yy, CFG['test_size'], CFG['seed'])
frac = CFG['calib_size'] / (1 - CFG['test_size'])
rest, calib_ids, y_rest, _ = strat_split(rest, y_rest, frac, CFG['seed'])
train_ids, val_ids, _, _ = strat_split(rest, y_rest, CFG['val_size'] / (1 - CFG['test_size'] - CFG['calib_size']), CFG['seed'])
split_of = {**{i: 'train' for i in train_ids}, **{i: 'val' for i in val_ids}, **{i: 'calib' for i in calib_ids}, **{i: 'test' for i in test_ids}}
df['split'] = df.Company_ID.map(split_of)
print(df.groupby('split').Fraud_Label.agg(['count', 'mean']).round(3))

tr = (df.split == 'train').values
num_raw = df[NUM_COLS].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
keep = [c for c in NUM_COLS if num_raw.loc[tr, c].notna().any()]               # drop all-NaN columns (train)
num_raw = num_raw[keep]
miss_cols = [c for c in keep if num_raw.loc[tr, c].isna().any()]
miss_ind = num_raw[miss_cols].isna().astype(np.float32).values
imputer = SimpleImputer(strategy='median').fit(num_raw[tr])
scaler = RobustScaler().fit(imputer.transform(num_raw[tr]))
X_scaled = np.clip(scaler.transform(imputer.transform(num_raw)), -10, 10).astype(np.float32)
X_NUM = np.hstack([X_scaled, miss_ind]).astype(np.float32)
NUM_FEATS = keep + [c + '__missing' for c in miss_cols]
ind_idx = {c: len(keep) + i for i, c in enumerate(miss_cols)}
GROUPS = {c: [j] + ([ind_idx[c]] if c in ind_idx else []) for j, c in enumerate(keep)}   # feature + its indicator
print('Numeric tensor:', X_NUM.shape, '| features with missingness indicators:', len(miss_cols))

## Stage 2 – Contribution 1: lightweight multimodal feature extraction
### 2.2 Frozen FinBERT embeddings (bounded-length chunks, mean-pooled). Falls back to TF-IDF+SVD if the model cannot be downloaded (e.g. internet disabled).

In [ ]:
@torch.no_grad()
def finbert_embed(texts, name, max_len, max_chunks, bs=16):
    from transformers import AutoTokenizer, AutoModel
    tok = AutoTokenizer.from_pretrained(name); tok.model_max_length = 10 ** 9
    enc = AutoModel.from_pretrained(name).to(DEVICE).eval()
    for p in enc.parameters(): p.requires_grad = False                 # frozen encoder
    body = max_len - 2
    out = np.zeros((len(texts), enc.config.hidden_size), np.float32)
    chunks, owner = [], []
    for i, t in enumerate(texts):
        if not t: continue
        ids_ = tok(t, add_special_tokens=False)['input_ids']
        for j in list(range(0, max(len(ids_), 1), body))[:max_chunks]:
            chunks.append(ids_[j:j + body]); owner.append(i)
    embs = []
    for b in range(0, len(chunks), bs):
        inp = [[tok.cls_token_id] + c + [tok.sep_token_id] for c in chunks[b:b + bs]]
        L = max(len(x) for x in inp)
        ids_t = torch.tensor([x + [tok.pad_token_id] * (L - len(x)) for x in inp]).to(DEVICE)
        att = torch.tensor([[1] * len(x) + [0] * (L - len(x)) for x in inp]).to(DEVICE)
        h = enc(input_ids=ids_t, attention_mask=att).last_hidden_state
        m = att.unsqueeze(-1).float()
        embs.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())       # token mean-pool inside chunk
    if embs:
        embs = np.vstack(embs); owner = np.array(owner)
        for i in np.unique(owner): out[i] = embs[owner == i].mean(0)   # mean over chunks
    del enc; torch.cuda.empty_cache()
    return out

def tfidf_svd_embed(texts, train_mask, dim=128):
    vec = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    vec.fit(np.array(texts, dtype=object)[train_mask]); Xs = vec.transform(texts)
    svd = TruncatedSVD(n_components=min(dim, Xs.shape[1] - 1), random_state=CFG['seed']).fit(Xs[train_mask])
    Z = svd.transform(Xs).astype(np.float32)
    return Z

TEXT_PRESENT = np.stack([(df[c + '__clean'].str.len() > 20).values for c in TEXT_COLS], 1)
emb_list, FB_OK = [], False
for c in TEXT_COLS:
    texts = df[c + '__clean'].tolist()
    key = hashlib.md5(('|'.join(texts) + CFG['finbert_name'] + str(CFG['max_chunks'])).encode()).hexdigest()[:10]
    cache = os.path.join(CFG['out_dir'], f'emb_{squash(c)}_{key}.npy')
    if os.path.exists(cache): E = np.load(cache); FB_OK = True
    else:
        E = None
        if CFG['use_finbert']:
            try:
                uniq = sorted(set(texts)); pos = {t: i for i, t in enumerate(uniq)}
                U = finbert_embed(uniq, CFG['finbert_name'], CFG['max_chunk_tokens'], CFG['max_chunks'])
                E = U[[pos[t] for t in texts]]; FB_OK = True; np.save(cache, E)
            except Exception as e:
                print('FinBERT unavailable -> TF-IDF+SVD fallback.', repr(e)[:120])
        if E is None: E = tfidf_svd_embed(texts, tr)
    emb_list.append(E)
    print(f'{c}: embedding {E.shape}')
print('Text encoder:', 'frozen FinBERT' if FB_OK else 'TF-IDF+SVD fallback')
text_emb = np.hstack(emb_list)
text_emb = StandardScaler().fit(text_emb[tr]).transform(text_emb).astype(np.float32)
text_emb[~np.hstack([np.repeat(TEXT_PRESENT[:, [i]], e.shape[1], axis=1) for i, e in enumerate(emb_list)])] = 0.0       # missing narrative -> zeros
X_TEXT = text_emb

### 2.3 TF-IDF lexical features (one TF-IDF + SVD block per lexical field; field/category embeddings are learned inside the model)

In [ ]:
SVD_DIM = CFG['svd_dim']
lex_blocks, lex_present, lex_vecs, lex_sparse = [], [], {}, []
for f in LEX_FIELDS:
    s = df[f].fillna('')
    present = (s.str.len() > 0).values
    vec = TfidfVectorizer(token_pattern=r'[^\s]+', lowercase=False, max_features=CFG['tfidf_max_features'], sublinear_tf=True)
    try: vec.fit(s[tr & present]); Xs = vec.transform(s)
    except ValueError: Xs = sparse.csr_matrix((len(s), 1)); vec = None
    lex_vecs[f] = vec; lex_sparse.append(Xs)
    k = min(SVD_DIM, Xs.shape[1] - 1, int((tr & present).sum()) - 1)
    Z = np.zeros((len(s), SVD_DIM), np.float32)
    if k >= 1:
        Z[:, :k] = TruncatedSVD(n_components=k, random_state=CFG['seed']).fit(Xs[tr & present]).transform(Xs)
    Z[~present] = 0
    lex_blocks.append(Z); lex_present.append(present)
X_LEX = np.stack(lex_blocks, 1).astype(np.float32)         # (N, F, svd)
X_LEXM = np.stack(lex_present, 1).astype(np.float32)       # (N, F)
LEX_TFIDF = sparse.hstack(lex_sparse).tocsr()              # full sparse lexical matrix for the linear baseline (Stage 7.2)
LEX_TERMS = np.concatenate([np.array(v.get_feature_names_out()) if v is not None else np.array(['<none>']) for v in lex_vecs.values()])
M_MOD = np.stack([np.ones(len(df)), TEXT_PRESENT.any(1), X_LEXM.any(1)], 1).astype(np.float32)   # [numeric, narrative, lexical]
print('Lexical tensor:', X_LEX.shape, '| modality availability (num, text, lex):', M_MOD.mean(0).round(3))

### 2.1 / 2.4 Model: FT-Transformer-lite (or MLP baseline) + projections + reliability-gated fusion + classification head

In [ ]:
class FTLite(nn.Module):
    """Compact feature-tokenisation transformer for numerical features."""
    def __init__(s, n_feat, d=16, layers=2, heads=2, out=64, drop=0.1):
        super().__init__()
        s.w = nn.Parameter(torch.randn(n_feat, d) * 0.02); s.b = nn.Parameter(torch.zeros(n_feat, d))
        s.cls = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=2 * d, dropout=drop, batch_first=True, norm_first=True)
        s.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        s.norm = nn.LayerNorm(d); s.out = nn.Linear(d, out)
    def forward(s, x):
        t = x.unsqueeze(-1) * s.w + s.b
        t = torch.cat([s.cls.expand(len(x), -1, -1), t], 1)
        return s.out(F.gelu(s.norm(s.enc(t)[:, 0])))

class MLPEnc(nn.Module):
    def __init__(s, n_feat, out=64, drop=0.1):
        super().__init__(); s.net = nn.Sequential(nn.Linear(n_feat, 128), nn.GELU(), nn.Dropout(drop), nn.Linear(128, out))
    def forward(s, x): return s.net(x)

class TextProj(nn.Module):
    def __init__(s, d_in, out=64, drop=0.2):
        super().__init__(); s.net = nn.Sequential(nn.Dropout(drop), nn.Linear(d_in, out), nn.LayerNorm(out), nn.GELU())
    def forward(s, x): return s.net(x)

class LexEnc(nn.Module):
    """Per-field projection + learned category (field) embedding, masked mean over available fields."""
    def __init__(s, svd, n_fields, out=64):
        super().__init__(); s.proj = nn.Sequential(nn.Linear(svd, out), nn.GELU()); s.cat = nn.Embedding(n_fields, out); s.norm = nn.LayerNorm(out)
    def forward(s, x, fm):
        h = (s.proj(x) + s.cat.weight.unsqueeze(0)) * fm.unsqueeze(-1)
        return s.norm(h.sum(1) / fm.sum(1, keepdim=True).clamp(min=1))

class GatedFusion(nn.Module):
    """Availability-masked softmax gate over modality latents."""
    def __init__(s, d, M=3):
        super().__init__(); s.gate = nn.Sequential(nn.Linear(M * d, d), nn.ReLU(), nn.Linear(d, M))
    def forward(s, hs, m):
        hs = hs * m.unsqueeze(-1)
        g = torch.softmax(s.gate(hs.flatten(1)).masked_fill(m == 0, -1e9), -1)
        return (g.unsqueeze(-1) * hs).sum(1), g

class FraudNet(nn.Module):
    def __init__(s, n_num, d_text, n_lex_fields, svd, enc='ft'):
        super().__init__(); d = CFG['d_model']
        s.enc_num = FTLite(n_num, CFG['ft_dim'], CFG['ft_layers'], CFG['ft_heads'], d) if enc == 'ft' else MLPEnc(n_num, d)
        s.enc_text = TextProj(d_text, d); s.enc_lex = LexEnc(svd, n_lex_fields, d)
        s.fusion = GatedFusion(d)
        s.head = nn.Sequential(nn.Linear(d, 32), nn.GELU(), nn.Dropout(0.2), nn.Linear(32, 1))
    def forward(s, xn, xt, xl, xlm, m):
        hs = torch.stack([s.enc_num(xn), s.enc_text(xt), s.enc_lex(xl, xlm)], 1)
        z, g = s.fusion(hs, m)
        return s.head(z).squeeze(-1), g

def build_model(enc='ft'):
    return FraudNet(X_NUM.shape[1], X_TEXT.shape[1], X_LEX.shape[1], X_LEX.shape[2], enc).to(DEVICE)
_m = build_model()
print('Trainable parameters (FT-lite model): %d' % sum(p.numel() for p in _m.parameters()))
print('Trainable parameters (MLP variant)  : %d' % sum(p.numel() for p in build_model('mlp').parameters()))
del _m

## Stage 3 – Federated client initialisation (company-level, label-skewed non-IID partition)

In [ ]:
def T(a, dt=torch.float32): return torch.tensor(a, dtype=dt, device=DEVICE)
def make_split(name):
    ix = np.where(df.split.values == name)[0]
    return dict(ix=ix, xn=T(X_NUM[ix]), xt=T(X_TEXT[ix]), xl=T(X_LEX[ix]), xlm=T(X_LEXM[ix]), m=T(M_MOD[ix]), y=T(df.Fraud_Label.values[ix]),
                cid=df.Company_ID.values[ix])
D = {s: make_split(s) for s in ['train', 'val', 'calib', 'test']}

def partition_clients(cids, y, K, alpha, seed):
    """Dirichlet label-skew over *companies*; every row of a company stays on one client."""
    rng = np.random.RandomState(seed)
    comp_df = pd.DataFrame({'c': cids, 'y': y}).groupby('c').y.max()
    cl = [[] for _ in range(K)]
    for lab in [0, 1]:
        members = rng.permutation(comp_df.index[comp_df.values == lab])
        p = rng.dirichlet([alpha] * K); cuts = (np.cumsum(p) * len(members)).astype(int)[:-1]
        for k, part in enumerate(np.split(members, cuts)): cl[k] += list(part)
    for k in range(K):                                         # guarantee a minimum size
        if len(cl[k]) < 8:
            donor = int(np.argmax([len(c) for c in cl])); take = cl[donor][:8]; cl[donor] = cl[donor][8:]; cl[k] += take
    return cl

parts = partition_clients(D['train']['cid'], D['train']['y'].cpu().numpy(), CFG['n_clients'], CFG['noniid_alpha'], CFG['seed'])
rng_g = np.random.RandomState(CFG['seed'])
def make_client(k, comp_list):
    sel = np.where(np.isin(D['train']['cid'], comp_list))[0]
    sub = {n: (v[torch.tensor(sel, device=DEVICE)] if torch.is_tensor(v) else v[sel]) for n, v in D['train'].items()}
    if CFG['simulate_modality_gaps']:                          # heterogeneous modality availability across clients
        r_text = CFG['text_gap_rates'][k % len(CFG['text_gap_rates'])]; r_lex = r_text / 2
        dt = torch.tensor(rng_g.rand(len(sel)) < r_text, device=DEVICE); dl = torch.tensor(rng_g.rand(len(sel)) < r_lex, device=DEVICE)
        sub['xt'] = sub['xt'].clone(); sub['xt'][dt] = 0; sub['m'] = sub['m'].clone(); sub['m'][dt, 1] = 0
        sub['xl'] = sub['xl'].clone(); sub['xl'][dl] = 0; sub['xlm'] = sub['xlm'].clone(); sub['xlm'][dl] = 0; sub['m'][dl, 2] = 0
    return sub
CLIENTS = [make_client(k, c) for k, c in enumerate(parts)]
print(pd.DataFrame([{'client': k, 'companies': len(parts[k]), 'rows': len(c['y']), 'fraud_rate': round(float(c['y'].mean()), 3),
                     'text_avail': round(float(c['m'][:, 1].mean()), 2), 'lex_avail': round(float(c['m'][:, 2].mean()), 2)} for k, c in enumerate(CLIENTS)]))

## Stage 4 – Local training (AdamW, class-weighted BCE, component-wise updates)

In [ ]:
POS_RATE = float(D['train']['y'].mean())
POS_W = (1 - POS_RATE) / max(POS_RATE, 1e-6) if POS_RATE < 0.40 else 1.0          # class weighting only if warranted
print('Train fraud rate %.3f -> pos_weight %.2f' % (POS_RATE, POS_W))
COMPONENTS = ['enc_num', 'enc_text', 'enc_lex', 'fusion', 'head']
comp_of = lambda k: k.split('.')[0]
def get_state(model): return {k: v.detach().clone() for k, v in model.state_dict().items()}

def safe_auc(y, p):
    try: return float(roc_auc_score(y, p))
    except ValueError: return 0.5

@torch.no_grad()
def logits_of(model, d, bs=512, **ov):
    model.eval(); out = []
    g = {k: ov.get(k, d[k]) for k in ['xn', 'xt', 'xl', 'xlm', 'm']}
    for i in range(0, len(g['xn']), bs):
        out.append(model(*[g[k][i:i + bs] for k in ['xn', 'xt', 'xl', 'xlm', 'm']])[0])
    return torch.cat(out).cpu().numpy()

def sub_dict(d, idx):
    idx = torch.tensor(idx, device=DEVICE)
    return {k: (v[idx] if torch.is_tensor(v) else v) for k, v in d.items() if k in ['xn', 'xt', 'xl', 'xlm', 'm', 'y']}

def local_update(gstate, cd, enc, seed):
    g = torch.Generator().manual_seed(seed); rs = np.random.RandomState(seed)
    model = build_model(enc); model.load_state_dict(gstate)
    # local validation reliability set (company-grouped)
    cids = np.unique(cd['cid']); rs.shuffle(cids); nv = max(1, int(0.15 * len(cids)))
    is_val = np.isin(cd['cid'], cids[:nv])
    trd, vld = sub_dict(cd, np.where(~is_val)[0]), sub_dict(cd, np.where(is_val)[0])
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_W, device=DEVICE))
    vloss = lambda: float(lossf(torch.tensor(logits_of(model, vld), device=DEVICE), vld['y']))
    lb = vloss()
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    n = len(trd['y'])
    for ep in range(CFG['local_epochs']):
        model.train(); perm = torch.randperm(n, generator=g).to(DEVICE)
        for i in range(0, n, CFG['batch']):
            b = perm[i:i + CFG['batch']]
            if len(b) < 2: continue
            m = trd['m'][b].clone()
            drop = (torch.rand(len(b), 2, device=DEVICE) < CFG['mod_dropout']).float()      # modality dropout (text, lexical)
            m[:, 1:] = m[:, 1:] * (1 - drop)
            logit, _ = model(trd['xn'][b], trd['xt'][b], trd['xl'][b], trd['xlm'][b], m)
            loss = lossf(logit, trd['y'][b])
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    new = get_state(model)
    delta = {k: new[k] - gstate[k] for k in new}
    lv = logits_of(model, vld)
    meta = dict(n=int(n), val_auc=safe_auc(vld['y'].cpu().numpy(), lv), val_loss_before=lb, val_loss_after=vloss(),
                norm=float(torch.sqrt(sum((v ** 2).sum() for v in delta.values()))),
                avail_text=float(cd['m'][:, 1].mean()), avail_lex=float(cd['m'][:, 2].mean()))
    return delta, meta

## Stage 5 – Contribution 2: reliability-adaptive, modality-aware aggregation with weight clipping
For client *i* and component *c*:
`w_ic ∝ log(1+n_i) · stability_i · validation_reliability_i · availability_ic`, then clipped at `w_clip` and renormalised.
* **stability** = `exp(-max(0, ‖Δ_i‖/median‖Δ‖ − 1))`, damped further if local validation loss worsens;
* **validation reliability** = `clip(2·(AUC_i − 0.5), 0.05, 1)`;
* **availability** is 1 for numerics, the client's text / lexical availability for those branches, and the mean of the three for fusion + head;
* `log(1+n)` instead of `n` prevents clients from dominating merely because they own more records.

In [ ]:
def clip_norm(w, cap):
    w = np.asarray(w, float); w = w / w.sum(); K = int((w > 0).sum()); cap = max(cap, 1.0 / max(K, 1) + 1e-6)
    for _ in range(100):
        over = w > cap
        if not over.any(): break
        excess = (w[over] - cap).sum(); w[over] = cap
        under = (~over) & (w > 0)
        if not under.any(): break
        w[under] += excess * w[under] / w[under].sum()
    return w / w.sum()

def reliability_weights(metas):
    norms = np.array([m['norm'] for m in metas]); med = np.median(norms) + 1e-12
    stab = np.exp(-np.maximum(0, norms / med - 1))
    stab = stab / (1 + np.maximum(0, np.array([m['val_loss_after'] - m['val_loss_before'] for m in metas])))
    vrel = np.clip(2 * (np.array([m['val_auc'] for m in metas]) - 0.5), 0.05, 1.0)
    n = np.array([m['n'] for m in metas], float)
    at, al = np.array([m['avail_text'] for m in metas]), np.array([m['avail_lex'] for m in metas])
    avail = {'enc_num': np.ones(len(metas)), 'enc_text': at, 'enc_lex': al, 'fusion': (1 + at + al) / 3, 'head': (1 + at + al) / 3}
    W = {}
    for c in COMPONENTS:
        raw = np.log1p(n) * stab * vrel * avail[c]; raw[avail[c] < 0.05] = 0
        if raw.sum() <= 0: raw = np.ones(len(metas))
        W[c] = clip_norm(raw, CFG['w_clip'])
    return W

def aggregate(gstate, deltas, metas, mode):
    if mode == 'fedavg':
        n = np.array([m['n'] for m in metas], float); W = {c: n / n.sum() for c in COMPONENTS}
    else:
        W = reliability_weights(metas)
    new = {}
    for k, v in gstate.items():
        w = W[comp_of(k)]
        new[k] = v + sum(float(w[i]) * deltas[i][k] for i in range(len(deltas)))
    return new, W

def evaluate_ap(state, split, enc):
    m = build_model(enc); m.load_state_dict(state); lg = logits_of(m, D[split]); y = D[split]['y'].cpu().numpy()
    return average_precision_score(y, lg), safe_auc(y, lg)

def run_federated(clients, mode, enc='ft', rounds=None, tag=''):
    rounds = rounds or CFG['rounds']; seed_all(CFG['seed'])
    gstate = get_state(build_model(enc))                       # Stage 3.2: shared initialisation
    best, best_ap, bad, hist, Wlog = None, -1, 0, [], []
    for r in range(1, rounds + 1):
        deltas, metas = [], []
        for k, cd in enumerate(clients):
            d, mt = local_update(gstate, cd, enc, seed=CFG['seed'] * 1000 + r * 10 + k)
            deltas.append(d); metas.append(mt)
        gstate, W = aggregate(gstate, deltas, metas, mode)
        ap, auc = evaluate_ap(gstate, 'val', enc)
        hist.append(dict(round=r, val_PR_AUC=ap, val_ROC_AUC=auc)); Wlog.append({c: W[c].round(3).tolist() for c in COMPONENTS})
        if ap > best_ap + 1e-4: best_ap, best, bad = ap, copy.deepcopy(gstate), 0
        else: bad += 1
        if r % 5 == 0 or r == 1: print(f'[{tag}] round {r:02d}  val PR-AUC {ap:.4f}  ROC-AUC {auc:.4f}')
        if bad >= CFG['patience']: print(f'[{tag}] early stop at round {r}'); break
    return best, pd.DataFrame(hist), Wlog

### Train: centralised baselines, FedAvg, and the proposed reliability-adaptive federation

In [ ]:
CENTRAL = [{**D['train'], 'cid': D['train']['cid']}]
res = {}
res['Centralised MLP'] = run_federated(CENTRAL, 'fedavg', 'mlp', rounds=CFG['rounds'], tag='central-MLP')
res['Centralised FT-lite'] = run_federated(CENTRAL, 'fedavg', 'ft', rounds=CFG['rounds'], tag='central-FT')
res['Federated FedAvg (FT-lite)'] = run_federated(CLIENTS, 'fedavg', 'ft', tag='FedAvg')
res['Federated Proposed (FT-lite)'] = run_federated(CLIENTS, 'reliability', 'ft', tag='Reliability-adaptive')
ENC_OF = {'Centralised MLP': 'mlp'}
STATE = {k: v[0] for k, v in res.items()}
FINAL = 'Federated Proposed (FT-lite)'
fig, ax = plt.subplots(figsize=(7, 4))
for k, v in res.items(): ax.plot(v[1]['round'], v[1]['val_PR_AUC'], marker='o', ms=3, label=k)
ax.set_xlabel('round / epoch-block'); ax.set_ylabel('validation PR-AUC'); ax.legend(fontsize=8); ax.grid(alpha=.3); plt.tight_layout()
plt.savefig(f"{CFG['out_dir']}/training_curves.png", dpi=150); plt.show()
print('Aggregation weights, last round (rows=components, cols=clients):')
display(pd.DataFrame(res[FINAL][2][-1]).T.rename(columns=lambda i: f'client{i}'))

## Stage 6 – Fraud prediction, temperature scaling and risk thresholding

In [ ]:
def get_model(name):
    m = build_model(ENC_OF.get(name, 'ft')); m.load_state_dict(STATE[name]); return m

def ece_score(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1); e = 0
    for i in range(bins):
        s = (p > edges[i]) & (p <= edges[i + 1]) if i else (p >= edges[0]) & (p <= edges[1])
        if s.any(): e += s.mean() * abs(y[s].mean() - p[s].mean())
    return float(e)

def fit_temperature(logits, y):
    lg, yt = torch.tensor(logits, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)
    logT = torch.zeros(1, requires_grad=True); opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=200)
    def closure():
        opt.zero_grad(); l = F.binary_cross_entropy_with_logits(lg / logT.exp(), yt); l.backward(); return l
    opt.step(closure); return float(logT.exp().clamp(0.05, 20))

sig = lambda z: 1 / (1 + np.exp(-z))
def metric_row(y, p, t):
    pred = (p >= t).astype(int)
    return dict(F1=f1_score(y, pred, zero_division=0), Precision=precision_score(y, pred, zero_division=0), Recall=recall_score(y, pred, zero_division=0),
                PR_AUC=average_precision_score(y, p), ROC_AUC=safe_auc(y, p), MCC=matthews_corrcoef(y, pred) if len(set(pred)) > 1 else 0.0,
                Brier=brier_score_loss(y, p), ECE=ece_score(y, p))

def best_f1_threshold(y, p):
    ts = np.linspace(0.05, 0.95, 91); return float(ts[np.argmax([f1_score(y, (p >= t).astype(int), zero_division=0) for t in ts])])

def cost_threshold(y, p, c_fn, c_fp):
    ts = np.linspace(0.05, 0.95, 91)
    cost = [c_fn * ((p < t) & (y == 1)).sum() + c_fp * ((p >= t) & (y == 0)).sum() for t in ts]
    return float(ts[int(np.argmin(cost))]), ts, np.array(cost)

rows = []
for name in STATE:
    mdl = get_model(name)
    lc, yc = logits_of(mdl, D['calib']), D['calib']['y'].cpu().numpy()
    Tm = fit_temperature(lc, yc)
    pv, yv = sig(logits_of(mdl, D['val']) / Tm), D['val']['y'].cpu().numpy()
    th = best_f1_threshold(yv, pv)
    pt, yt = sig(logits_of(mdl, D['test']) / Tm), D['test']['y'].cpu().numpy()
    rows.append({'Model': name, 'T': round(Tm, 3), 'thr(val-F1)': th, **{k: round(v, 4) for k, v in metric_row(yt, pt, th).items()}})
RESULTS = pd.DataFrame(rows).set_index('Model'); display(RESULTS)
RESULTS.to_csv(f"{CFG['out_dir']}/model_comparison.csv")

# ---- final model: calibration, cost-sensitive threshold
FM = get_model(FINAL)
lg_cal, y_cal = logits_of(FM, D['calib']), D['calib']['y'].cpu().numpy()
TEMP = fit_temperature(lg_cal, y_cal)
p_raw_cal, p_cal_cal = sig(lg_cal), sig(lg_cal / TEMP)
print(f'Temperature T = {TEMP:.3f} | calibration-set ECE: {ece_score(y_cal, p_raw_cal):.4f} -> {ece_score(y_cal, p_cal_cal):.4f}')
p_val = sig(logits_of(FM, D['val']) / TEMP); y_val = D['val']['y'].cpu().numpy()
THR, ts_grid, cost_curve = cost_threshold(y_val, p_val, CFG['cost_fn'], CFG['cost_fp'])
print(f"Decision threshold (cost FN:FP = {CFG['cost_fn']}:{CFG['cost_fp']}) = {THR:.2f}   |   max-F1 threshold = {best_f1_threshold(y_val, p_val):.2f}")

p_test = sig(logits_of(FM, D['test']) / TEMP); y_test = D['test']['y'].cpu().numpy()
FINAL_METRICS = metric_row(y_test, p_test, THR); print('Test metrics @ cost-optimal threshold:', {k: round(v, 4) for k, v in FINAL_METRICS.items()})
print('Confusion matrix [[TN FP],[FN TP]]:\n', confusion_matrix(y_test, (p_test >= THR).astype(int), labels=[0, 1]))

p_test_raw = sig(logits_of(FM, D['test']))
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for p_, lab in [(p_test_raw, 'before'), (p_test, 'after temperature scaling')]:
    edges = np.linspace(0, 1, 9); idx = np.digitize(p_, edges[1:-1]); xs, ys = [], []
    for b in range(8):
        if (idx == b).sum() >= 3: xs.append(p_[idx == b].mean()); ys.append(y_test[idx == b].mean())
    ax[0].plot(xs, ys, marker='o', label=lab)
ax[0].plot([0, 1], [0, 1], 'k--'); ax[0].set_title('Reliability diagram (test)'); ax[0].set_xlabel('predicted'); ax[0].set_ylabel('observed'); ax[0].legend()
pr, rc, _ = precision_recall_curve(y_test, p_test); ax[1].plot(rc, pr); ax[1].set_title('PR curve (AP=%.3f)' % FINAL_METRICS['PR_AUC']); ax[1].set_xlabel('recall'); ax[1].set_ylabel('precision')
ax[2].plot(ts_grid, cost_curve); ax[2].axvline(THR, color='r', ls='--'); ax[2].set_title('Validation expected cost vs threshold'); ax[2].set_xlabel('threshold')
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/calibration_threshold.png", dpi=150); plt.show()

## Stage 7 – Lightweight explainability

In [ ]:
def calibrated_p(model, d, **ov): return sig(logits_of(model, d, **ov) / TEMP)

# ---- 7.1 permutation importance (feature + its missing indicator permuted together)
def perm_importance(model, d, n_rep=5, seed=0):
    rng = np.random.RandomState(seed); y = d['y'].cpu().numpy(); base = average_precision_score(y, calibrated_p(model, d)); out = {}
    for name, cols in GROUPS.items():
        drops = []
        for _ in range(n_rep):
            X = d['xn'].clone(); pm = torch.tensor(rng.permutation(len(X)), device=DEVICE)
            X[:, cols] = X[pm][:, cols]
            drops.append(base - average_precision_score(y, calibrated_p(model, d, xn=X)))
        out[name] = (np.mean(drops), np.std(drops))
    return pd.DataFrame(out, index=['mean_PR_AUC_drop', 'std']).T.sort_values('mean_PR_AUC_drop', ascending=False), base
PI, base_ap = perm_importance(FM, D['test'])
print('Baseline test PR-AUC: %.4f' % base_ap); display(PI.head(10))

# ---- 7.2 linear lexical / text baselines
tr_idx, te_idx = np.where(df.split.values == 'train')[0], np.where(df.split.values == 'test')[0]
lr_lex = LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000).fit(LEX_TFIDF[tr_idx], df.Fraud_Label.values[tr_idx])
p_lex = lr_lex.predict_proba(LEX_TFIDF[te_idx])[:, 1]
print('Linear lexical baseline (test): PR-AUC %.4f | ROC-AUC %.4f' % (average_precision_score(y_test, p_lex), safe_auc(y_test, p_lex)))
coef = pd.Series(lr_lex.coef_[0], index=LEX_TERMS[:LEX_TFIDF.shape[1]]).sort_values()
top_terms = pd.concat([coef.head(8), coef.tail(8)]); display(top_terms.rename('coef'))

nar_vec = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xn_tr = nar_vec.fit_transform(all_text.values[tr_idx]); lr_txt = LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000).fit(Xn_tr, df.Fraud_Label.values[tr_idx])
p_txt = lr_txt.predict_proba(nar_vec.transform(all_text.values[te_idx]))[:, 1]
print('Linear narrative baseline (test): PR-AUC %.4f | ROC-AUC %.4f' % (average_precision_score(y_test, p_txt), safe_auc(y_test, p_txt)))
TXT_COEF = dict(zip(nar_vec.get_feature_names_out(), lr_txt.coef_[0]))

# ---- 7.3 modality ablation (F1 / PR-AUC change on test)
def ablate(model, d, drop):
    ov = {}; m = d['m'].clone()
    if 'numerical' in drop: ov['xn'] = torch.zeros_like(d['xn']); m[:, 0] = 0
    if 'narrative' in drop: ov['xt'] = torch.zeros_like(d['xt']); m[:, 1] = 0
    if 'lexical' in drop: ov['xl'] = torch.zeros_like(d['xl']); ov['xlm'] = torch.zeros_like(d['xlm']); m[:, 2] = 0
    ov['m'] = m; return calibrated_p(model, d, **ov)
abl = [('Full model', [])] + [(f'- {n}', [n]) for n in ['numerical', 'narrative', 'lexical']] + [('numerical only', ['narrative', 'lexical']), ('narrative only', ['numerical', 'lexical']), ('lexical only', ['numerical', 'narrative'])]
ab_rows = []
for nm, drop in abl:
    p_ = ablate(FM, D['test'], drop); r = metric_row(y_test, p_, THR); ab_rows.append({'Setting': nm, 'F1': r['F1'], 'PR_AUC': r['PR_AUC']})
AB = pd.DataFrame(ab_rows).set_index('Setting'); AB['dF1'] = AB.F1 - AB.loc['Full model', 'F1']; AB['dPR_AUC'] = AB.PR_AUC - AB.loc['Full model', 'PR_AUC']
display(AB.round(4)); AB.to_csv(f"{CFG['out_dir']}/modality_ablation.csv")
with torch.no_grad():
    gates = FM(D['test']['xn'], D['test']['xt'], D['test']['xl'], D['test']['xlm'], D['test']['m'])[1].cpu().numpy()
print('Mean fusion gate (numerical, narrative, lexical):', gates.mean(0).round(3))

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
PI.head(10)['mean_PR_AUC_drop'][::-1].plot.barh(ax=ax[0], xerr=PI.head(10)['std'][::-1]); ax[0].set_title('7.1 Permutation importance (PR-AUC drop)')
top_terms.plot.barh(ax=ax[1], color=['tab:blue' if v < 0 else 'tab:red' for v in top_terms]); ax[1].set_title('7.2 Lexical LR coefficients (+ = fraud)')
AB.loc[[f'- {n}' for n in ['numerical', 'narrative', 'lexical']], ['dF1', 'dPR_AUC']].plot.bar(ax=ax[2]); ax[2].set_title('7.3 Modality ablation (change vs full)'); ax[2].axhline(0, color='k', lw=.5)
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/explainability.png", dpi=150); plt.show()

## Stage 8 – Contribution 3: uncertainty-aware financial risk assessment
### 8.1 Risk ranking · 8.2 Uncertainty screening · 8.3 Risk reporting

In [ ]:
LO, HI = CFG['band']
risk_level = lambda p: 'High' if p >= CFG['high_cut'] else ('Medium' if p >= CFG['med_cut'] else 'Low')
def status(p): return 'Uncertain (Manual Review)' if LO <= p <= HI else ('High risk (confident)' if p > HI else 'Low risk (confident)')

all_rows = []
for s in ['train', 'val', 'calib', 'test']:
    p = calibrated_p(FM, D[s])
    all_rows.append(pd.DataFrame({'Company_ID': D[s]['cid'], 'row': D[s]['ix'], 'Fraud_Probability': p, 'split': s, 'Fraud_Label': D[s]['y'].cpu().numpy()}))
ALL = pd.concat(all_rows)
RANK_ALL = ALL.groupby('Company_ID').agg(Fraud_Probability=('Fraud_Probability', 'mean'), split=('split', 'first'),
                                        Fraud_Label=('Fraud_Label', 'max')).reset_index()
RANK_ALL = RANK_ALL.sort_values('Fraud_Probability', ascending=False).reset_index(drop=True)
RANK_ALL.insert(0, 'Rank', np.arange(1, len(RANK_ALL) + 1))
RANK_ALL['Risk_Level'] = RANK_ALL.Fraud_Probability.map(risk_level)
RANK_ALL['Status'] = RANK_ALL.Fraud_Probability.map(status)
RANK_ALL['Predicted_Label'] = (RANK_ALL.Fraud_Probability >= THR).astype(int)
RANK_ALL.to_csv(f"{CFG['out_dir']}/fraud_risk_ranking_all_companies.csv", index=False)
RANK = RANK_ALL[RANK_ALL.split == 'test'].reset_index(drop=True); RANK['Rank'] = np.arange(1, len(RANK) + 1)
print('Held-out (test) companies ranked:', len(RANK)); display(RANK.head(10)[['Rank', 'Company_ID', 'Fraud_Probability', 'Risk_Level', 'Status', 'Fraud_Label']].round(3))

UNC = RANK[RANK.Status.str.startswith('Uncertain')]; CONF_HIGH = RANK[RANK.Status.str.startswith('High')]
print(f'Alerts on test set -> high-risk confident: {len(CONF_HIGH)} | uncertain (manual review, {LO}-{HI}): {len(UNC)} | low-risk confident: {len(RANK) - len(UNC) - len(CONF_HIGH)}')
def prec_at_k(k): r = RANK.head(k); return r.Fraud_Label.mean()
print('Precision@10 %.3f | Precision@20 %.3f | base rate %.3f' % (prec_at_k(10), prec_at_k(20), RANK.Fraud_Label.mean()))
conf = RANK[~RANK.Status.str.startswith('Uncertain')]
print('Accuracy on confident predictions: %.3f (n=%d)  vs on uncertain cases: %.3f (n=%d)' % (
    ((conf.Fraud_Probability >= THR).astype(int) == conf.Fraud_Label).mean() if len(conf) else float('nan'), len(conf),
    ((UNC.Fraud_Probability >= THR).astype(int) == UNC.Fraud_Label).mean() if len(UNC) else float('nan'), len(UNC)))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(ALL.groupby('Company_ID').Fraud_Probability.mean(), bins=[0, .2, .4, .6, .8, 1.0], edgecolor='k')
ax[0].axvspan(LO, HI, color='orange', alpha=.25, label='uncertain band'); ax[0].axvline(THR, color='r', ls='--', label=f'threshold {THR:.2f}')
ax[0].set_xlabel('fraud probability'); ax[0].set_ylabel('companies'); ax[0].set_title('Fraud-risk distribution (all companies)'); ax[0].legend()
RANK.Risk_Level.value_counts().reindex(['High', 'Medium', 'Low']).plot.bar(ax=ax[1], color=['tab:red', 'tab:orange', 'tab:green']); ax[1].set_title('Risk levels (test)')
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/risk_distribution.png", dpi=150); plt.show()
print('Uncertain cases (manual review):'); display(UNC[['Company_ID', 'Fraud_Probability', 'Status']].head(10).round(3))

### 8.3 Interpretable company-level risk reports
Numerical drivers come from per-company occlusion (feature set to the training median → change in calibrated probability); text evidence = narrative sentences with the highest linear-model fraud evidence; lexical terms come from the lexical logistic baseline.

In [ ]:
SENT_SPLIT = re.compile(r'(?<=[\.\!\?])\s+')
def numeric_drivers(row_pos, d, topk=5):
    base = calibrated_p(FM, {k: v[row_pos:row_pos + 1] for k, v in d.items() if torch.is_tensor(v)})[0]
    n = len(GROUPS); X = d['xn'][row_pos:row_pos + 1].repeat(n, 1)
    for i, cols in enumerate(GROUPS.values()): X[i, cols] = 0.0              # median in scaled space, indicator off
    rep = {k: d[k][row_pos:row_pos + 1].repeat(n, *[1] * (d[k].dim() - 1)) for k in ['xt', 'xl', 'xlm', 'm']}
    p_occ = sig(logits_of(FM, {'xn': X, **rep}) / TEMP)
    contrib = pd.Series(base - p_occ, index=list(GROUPS.keys())).sort_values(key=np.abs, ascending=False)
    return contrib.head(topk)
def text_evidence(row_idx, topn=2):
    scored = []
    for c in TEXT_COLS:
        for s in SENT_SPLIT.split(df.loc[row_idx, c + '__clean']):
            toks = re.findall(r"[a-z0-9\$%']+", s)
            if len(toks) < 4: continue
            bi = [' '.join(toks[i:i + 2]) for i in range(len(toks) - 1)]
            sc = (sum(TXT_COEF.get(t, 0) for t in toks) + sum(TXT_COEF.get(b, 0) for b in bi)) / math.sqrt(len(toks))
            scored.append((sc, c, s))
    return sorted(scored, key=lambda x: -x[0])[:topn]
def lexical_terms(row_idx, topn=4):
    present = set(' '.join(df.loc[row_idx, LEX_FIELDS].astype(str)).split()); sub = coef[coef.index.isin(present)]
    return list(sub.sort_values(ascending=False).head(topn).index)

def company_report(cid, store=None):
    r = RANK[RANK.Company_ID == cid].iloc[0]
    rows_c = ALL[(ALL.Company_ID == cid) & (ALL.split == 'test')].sort_values('Fraud_Probability', ascending=False)
    row_global = int(rows_c.iloc[0]['row']); pos = int(np.where(D['test']['ix'] == row_global)[0][0])
    drivers = numeric_drivers(pos, D['test']); ev = text_evidence(row_global); lt = lexical_terms(row_global)
    rep = dict(Company_ID=cid, Rank=int(r.Rank), Fraud_Probability=round(float(r.Fraud_Probability), 3), Risk_Level=r.Risk_Level, Status=r.Status,
               Top_Numerical_Features={k: round(float(v), 3) for k, v in drivers.items()},
               Key_Text_Evidence=[{'field': f, 'sentence': s[:240]} for _, f, s in ev], Key_Lexical_Terms=lt)
    return rep
def print_report(rep):
    print('=' * 78); print(f"Company {rep['Company_ID']} | rank {rep['Rank']} | {rep['Risk_Level']} risk | {rep['Status']}"); print(f"Fraud probability: {rep['Fraud_Probability']}")
    print('Top numerical drivers (change in probability when feature is neutralised; + = raises risk):')
    for k, v in rep['Top_Numerical_Features'].items(): print(f'   {k:35s} {v:+.3f}')
    print('Key text evidence:'); [print(f"   [{e['field']}] \"...{e['sentence']}...\"") for e in rep['Key_Text_Evidence']]
    print('Key lexical terms:', ', '.join(rep['Key_Lexical_Terms']) or '-')

ALERTS = pd.concat([CONF_HIGH, UNC]).sort_values('Fraud_Probability', ascending=False)
REPORTS = [company_report(c) for c in ALERTS.Company_ID.head(25)]
for rep in REPORTS[:3]: print_report(rep)
with open(f"{CFG['out_dir']}/risk_reports.json", 'w') as f: json.dump(REPORTS, f, indent=2, default=str)
ALERTS.to_csv(f"{CFG['out_dir']}/investigation_alerts.csv", index=False)
print('\nSaved:', os.listdir(CFG['out_dir']))

### 8.4 Optional investment integration (only if you supply historical returns)
The fraud dataset alone **cannot** validate stock-price prediction. Set `CFG['price_csv']` to a CSV with columns `Company_ID, Date, Return` (periodic simple returns) to compare an equal-weight baseline with a risk-aware portfolio that excludes confident-high-risk companies and halves exposure to uncertain ones.

In [ ]:
if CFG['price_csv'] and os.path.exists(CFG['price_csv']):
    px = pd.read_csv(CFG['price_csv'], parse_dates=['Date']); px['Company_ID'] = px.Company_ID.astype(str)
    R = px.pivot_table(index='Date', columns='Company_ID', values='Return').sort_index()
    univ = [c for c in R.columns if c in set(RANK_ALL.Company_ID)]; R = R[univ].fillna(0)
    st = RANK_ALL.set_index('Company_ID').loc[univ]
    w_base = np.ones(len(univ)) / len(univ)
    w_risk = np.where(st.Status.str.startswith('High'), 0.0, np.where(st.Status.str.startswith('Uncertain'), 0.5, 1.0)); w_risk = w_risk / w_risk.sum()
    def cvar(r, a=0.05): q = np.quantile(r, a); return float(r[r <= q].mean())
    out = {}
    for nm, w in [('Baseline', w_base), ('Risk-aware', w_risk)]:
        r = R.values @ w; out[nm] = dict(cum_return_pct=100 * (np.prod(1 + r) - 1), CVaR_5pct=cvar(r), vol=r.std())
    display(pd.DataFrame(out).T)
    pd.DataFrame({'Baseline': np.cumprod(1 + R.values @ w_base) - 1, 'Risk-aware': np.cumprod(1 + R.values @ w_risk) - 1}, index=R.index).mul(100).plot(figsize=(7, 4), title='Cumulative return (%)')
    plt.show()
else:
    print('Stage 8.4 skipped: no market data supplied (CFG["price_csv"] is None).')

## Summary of outputs
All artefacts are written to `CFG['out_dir']`: `model_comparison.csv`, `modality_ablation.csv`, `fraud_risk_ranking_all_companies.csv`, `investigation_alerts.csv`, `risk_reports.json`, and the figures.

In [ ]:
torch.save({'state': STATE[FINAL], 'temperature': TEMP, 'threshold': THR, 'num_features': NUM_FEATS, 'text_cols': TEXT_COLS, 'lex_fields': LEX_FIELDS},
           f"{CFG['out_dir']}/global_model.pt")
print('Final model | T = %.3f | threshold = %.2f' % (TEMP, THR)); display(RESULTS)